![image.png](https://i.imgur.com/a3uAqnb.png)

# ZeRO from Scratch: Sharding Model States Across Data-Parallel GPUs

- **Dataset**: [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) from Hugging Face, simple short stories; we use 20,000
- **Task**: character-level language modelling
- **Model**: a 4-layer character-level GPT (about 3.2M parameters), trained from scratch on simulated 4-GPU clusters
- **Runtime**: Colab T4 GPU, about 7 minutes of training

In this lab, we will:

- build an all-reduce from a ring reduce-scatter and all-gather, and count the bytes each GPU sends
- train one GPT three ways side by side and compare the weights bit for bit
- check what each GPU holds and sends against the formulas
- plan the memory of 7B and 70B models on 80 GB GPUs

## What is ZeRO?

In **data parallelism** (PyTorch's `DistributedDataParallel`, **DDP**), each of $d$ GPUs, run by a process called a **rank** ($0$ to $d-1$), holds a full **replica** of the model and computes the gradient on its own share of the batch, its **micro-batch**. An **all-reduce** sums the gradients onto every rank (divided by $d$, the sum is the whole batch's gradient), and all ranks take the same step. So every rank also stores the full gradient and optimizer state: for Adam, running averages of the gradient ($m$) and its square ($v$). **ZeRO** (Zero Redundancy Optimizer) cuts these into $d$ **chunks** and keeps chunk $r$, the rank's **shard**, on rank $r$ only, in three stages:

$$\begin{aligned} M_{\text{DDP}} &= (W+G+K)\,\Psi, & M_1 &= (W+G)\,\Psi + \frac{K\,\Psi}{d}, \\ M_2 &= W\,\Psi + \frac{(G+K)\,\Psi}{d}, & M_3 &= \frac{(W+G+K)\,\Psi}{d} \end{aligned}$$

Where:
- $M$ is the memory per rank for the weights, gradients and optimizer state (the **model states**), in bytes: stage 1 shards the optimizer state, stage 2 also the gradients, stage 3 also the weights
- $\Psi$ is the number of parameters and $d$ the number of ranks
- $W, G, K$ are bytes per parameter: $4, 4, 8$ here (32-bit floats); $2, 2, 12$ in mixed precision, which computes in 16 bits and keeps a 32-bit copy of the weights beside $m$ and $v$; both give DDP $16\Psi$

Stages 1 and 2 add no communication. The usual all-reduce is a **reduce-scatter**, leaving rank $r$ with the sum of chunk $r$ only, then an **all-gather** of the summed chunks. Adam updates each parameter from its own gradient, $m$ and $v$, so a ZeRO rank updates its chunk in between and all-gathers weights instead of gradients. Stage 3 keeps only its chunk of the weights, so it all-gathers them before the forward and again before the backward pass, not once after the update: three transfers instead of two, 1.5 times the traffic.

Paper: [Rajbhandari et al., 2020. ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054v3)

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom. With one GPU, we simulate the cluster: each rank is a Python object with its own tensors, and the **collectives**, operations every rank takes part in, copy real numbers between them. What a rank sends, and what it holds when it updates, are exact; only the timing is not.

In [ ]:
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
SEED = 42
N_TRAIN = 20_000          # training stories (the file we load has 529,930)
N_VAL = 500               # validation stories
CONTEXT = 128             # characters the model reads at once
N_LAYERS = 4
D_MODEL = 256
N_HEADS = 4
WORLD_SIZE = 4            # simulated GPUs (ranks) per cluster
MICRO_BATCH_SIZE = 16     # windows per rank per step; the global batch is WORLD_SIZE x MICRO_BATCH_SIZE
MAX_STEPS = 1_500
WARMUP_STEPS = 100
LR = 2e-3                 # peak learning rate
BETA1 = 0.9               # decay rate of Adam's m
BETA2 = 0.95              # decay rate of Adam's v
EPS = 1e-8
WEIGHT_DECAY = 0.1
EVAL_EVERY = 100          # steps between evaluations
MAX_NEW_TOKENS = 400      # characters to generate at the end

# Deterministic kernels give the same bits for the same inputs, so the three strategies can be
# compared exactly. cuBLAS reads this variable when PyTorch first uses it, before any matrix multiply.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.use_deterministic_algorithms(True)
# Deterministic mode also fills every new tensor with NaN, to expose reads of uninitialised memory;
# this lab never reads any, so we skip that extra work.
torch.utils.deterministic.fill_uninitialized_memory = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so training will be slow. In Colab: Runtime > Change runtime type > T4 GPU.")
# No mixed precision: every tensor is a 32-bit float (4 bytes per number), which keeps the byte counts simple.

## 1. Dataset

- **Source**: [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) (Eldan and Li, 2023), simple stories generated by GPT-3.5 and GPT-4
- **Content**: 2,119,719 training and 21,990 validation stories in a `text` column
- **Licence**: CDLA-Sharing-1.0
- **What we use**: 20,000 training and 500 validation stories from the first of four training files (249 MB) and the validation file, keeping only plain-ASCII stories

In [ ]:
# One training file and the validation file, at the revision this lab was tested with.
stories = load_dataset("roneneldan/TinyStories", revision="f54c09fd23315a6f9c86f9dc80f725de7d8f9c64",
                       data_files={"train": "data/train-00000-of-00004-2d5a1467fff1081b.parquet",
                                   "validation": "data/validation-00000-of-00001-869c898b519ad725.parquet"})
print(stories)


def ascii_stories(split, n):
    """The first n plain-ASCII stories of the split, in an order shuffled with SEED. About 6% of
    the stories contain punctuation garbled by an encoding mix-up (a curly quote stored as 'â€œ')."""
    candidates = split.shuffle(seed=SEED).select(range(min(len(split), 2 * n)))["text"]
    return [text for text in candidates if text.isascii()][:n]


train_texts = ascii_stories(stories["train"], N_TRAIN)
val_texts = ascii_stories(stories["validation"], N_VAL)
print(f"{len(train_texts):,} training and {len(val_texts):,} validation stories\n")
print(train_texts[0][:600])

Each character (newline or printable ASCII, 96 in all) becomes an id, and each split one sequence of ids. An example is a window of `CONTEXT` ids; its target is the window shifted by one.

In [ ]:
CHARS = ["\n"] + [chr(code) for code in range(32, 127)]   # newline and the 95 printable ASCII characters
VOCAB_SIZE = len(CHARS)
char_to_id = np.full(128, CHARS.index(" "), dtype=np.int64)   # any other ASCII character (a tab) becomes a space
for i, ch in enumerate(CHARS):
    char_to_id[ord(ch)] = i


def encode(texts):
    """List of strings -> one int64 tensor of character ids, the strings a blank line apart.
    A non-ASCII character (in a prompt you type) becomes '?'."""
    raw = np.frombuffer("\n\n".join(texts).encode("ascii", errors="replace"), dtype=np.uint8)
    return torch.from_numpy(char_to_id[raw])


def decode(ids):
    return "".join(CHARS[i] for i in ids)


def get_batch(data, batch_size, generator):
    """batch_size windows at random positions: inputs and next-character targets, each (batch_size, CONTEXT)."""
    starts = torch.randint(0, len(data) - CONTEXT, (batch_size,), generator=generator)
    windows = data[starts[:, None] + torch.arange(CONTEXT + 1)]          # (batch_size, CONTEXT + 1)
    return windows[:, :-1].to(DEVICE), windows[:, 1:].to(DEVICE)


train_data, val_data = encode(train_texts), encode(val_texts)
# Validation uses fixed, consecutive windows, so every evaluation sees the same characters.
val_starts = torch.arange(0, len(val_data) - CONTEXT, CONTEXT)
val_windows = val_data[val_starts[:, None] + torch.arange(CONTEXT + 1)]  # (n_windows, CONTEXT + 1)
val_inputs, val_targets = val_windows[:, :-1], val_windows[:, 1:]
print(f"vocabulary: {VOCAB_SIZE} characters | training text: {len(train_data):,} characters | "
      f"validation: {len(val_inputs):,} windows")

x, y = get_batch(train_data, 4, torch.Generator().manual_seed(SEED))
print("inputs", tuple(x.shape), "| targets", tuple(y.shape))            # (4, CONTEXT) each
print("input: ", repr(decode(x[0, :50].tolist())))
print("target:", repr(decode(y[0, :50].tolist())))

## 2. Model

A small GPT: character and position embeddings, `N_LAYERS` Transformer blocks (causal self-attention, then an MLP, each with a residual connection) and a linear layer scoring the next character. Attention uses plain matrix multiplies, not a fused kernel, so every GPU operation can run deterministically.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(D_MODEL, 3 * D_MODEL)
        self.proj = nn.Linear(D_MODEL, D_MODEL)
        # position i may attend to positions 0..i only
        self.register_buffer("causal", torch.tril(torch.ones(CONTEXT, CONTEXT, dtype=torch.bool)), persistent=False)

    def forward(self, x):                                                # x: (B, T, D)
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)                            # each (B, T, D)
        q, k, v = (t.view(B, T, N_HEADS, D // N_HEADS).transpose(1, 2) for t in (q, k, v))   # (B, H, T, D/H)
        scores = q @ k.transpose(-2, -1) / math.sqrt(D // N_HEADS)       # (B, H, T, T)
        attention = scores.masked_fill(~self.causal[:T, :T], float("-inf")).softmax(dim=-1)
        out = (attention @ v).transpose(1, 2).reshape(B, T, D)           # (B, T, D)
        return self.proj(out)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.attn = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.mlp = nn.Sequential(nn.Linear(D_MODEL, 4 * D_MODEL), nn.GELU(), nn.Linear(4 * D_MODEL, D_MODEL))

    def forward(self, x):                                                # (B, T, D) -> (B, T, D)
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class CharGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.pos_emb = nn.Parameter(torch.zeros(CONTEXT, D_MODEL))
        self.blocks = nn.ModuleList([Block() for _ in range(N_LAYERS)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)
        for name, p in self.named_parameters():                         # GPT-2's initialisation
            if p.dim() == 2:
                into_residual = name.endswith(("attn.proj.weight", "mlp.2.weight"))
                nn.init.normal_(p, std=0.02 / math.sqrt(2 * N_LAYERS) if into_residual else 0.02)
            elif name.endswith("bias"):
                nn.init.zeros_(p)

    def forward(self, idx):                                              # idx: (B, T) character ids
        x = self.tok_emb(idx) + self.pos_emb[:idx.shape[1]]              # (B, T, D)
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))                                   # (B, T, VOCAB_SIZE) logits


torch.manual_seed(SEED)
model = CharGPT().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
with torch.no_grad():
    print(model(x).shape)                                                # (4, CONTEXT, VOCAB_SIZE)

Two baselines for the validation loss, the cross-entropy in nats (natural-log units) per character: the untrained model, near $\ln 96 \approx 4.56$, and the training text's character frequencies, which ignore context.

In [ ]:
@torch.no_grad()
def evaluate(model, batch_size=64):
    """Mean cross-entropy (nats per character) of the model over all validation windows."""
    model.eval()
    total = 0.0
    for i in range(0, len(val_inputs), batch_size):
        logits = model(val_inputs[i:i + batch_size].to(DEVICE))
        targets = val_targets[i:i + batch_size].to(DEVICE)
        total += F.cross_entropy(logits.flatten(0, 1), targets.flatten(), reduction="sum").item()
    model.train()
    return total / val_targets.numel()


counts = torch.bincount(train_data, minlength=VOCAB_SIZE).double() + 1    # +1: no character gets probability 0
unigram_loss = -(counts / counts.sum())[val_targets.flatten()].log().mean().item()
untrained_loss = evaluate(model)
print(f"untrained model:       {untrained_loss:.3f} nats per character")
print(f"character frequencies: {unigram_loss:.3f} nats per character")

## 3. A Simulated Cluster

### One rank

A rank keeps its replica's weights in one **flat buffer**, a 1-D tensor holding every parameter end to end; each parameter becomes a view of its slice (a tensor sharing that memory), and the gradients fill a second flat buffer the same way. The buffer is padded to a multiple of $d$; rank $r$ owns chunk $r$.

In [ ]:
class Rank:
    """One simulated GPU: a model replica whose weights live in one flat buffer."""

    def __init__(self, rank, model, world_size, strategy):
        self.rank, self.model, self.strategy = rank, model, strategy
        self.params = list(model.parameters())
        n = sum(p.numel() for p in self.params)
        self.chunk_size = math.ceil(n / world_size)                      # numbers per chunk
        self.own = slice(rank * self.chunk_size, (rank + 1) * self.chunk_size)   # the chunk this rank owns
        self.flat_weights = torch.zeros(self.chunk_size * world_size, device=DEVICE)   # padded to a multiple of d
        offset = 0
        for p in self.params:                                            # each parameter becomes a view into the buffer
            self.flat_weights[offset:offset + p.numel()].copy_(p.detach().flatten())
            p.data = self.flat_weights[offset:offset + p.numel()].view_as(p)
            offset += p.numel()
        self.flat_grads = None                                           # the full gradient, while the rank holds it
        self.grad_shard = None                                           # ZeRO-2: the owned chunk of the gradient only
        # Adam's m and v: for the whole buffer under DDP, for the owned chunk only under ZeRO.
        state_size = len(self.flat_weights) if strategy == "ddp" else self.chunk_size
        self.exp_avg = torch.zeros(state_size, device=DEVICE)            # m
        self.exp_avg_sq = torch.zeros(state_size, device=DEVICE)         # v

    def forward_backward(self, inputs, targets):
        """Loss and gradient on this rank's micro-batch; the gradient lands in the flat gradient buffer."""
        if self.flat_grads is None:                                      # first step, or freed by ZeRO-2
            self.flat_grads = torch.zeros_like(self.flat_weights)
        else:
            self.flat_grads.zero_()
        offset = 0
        for p in self.params:                                            # backward adds each gradient into its view
            p.grad = self.flat_grads[offset:offset + p.numel()].view_as(p)
            offset += p.numel()
        self.grad_shard = None
        logits = self.model(inputs)                                      # (B, T, VOCAB_SIZE)
        # Keep the loss 2-D, (B*T, VOCAB_SIZE): the (B, VOCAB_SIZE, T) form has no deterministic GPU kernel.
        loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        loss.backward()
        return loss.detach()

    def keep_own_gradient_only(self):
        """ZeRO-2: keep the owned chunk of the summed gradient and free the full buffer."""
        self.grad_shard = self.flat_grads[self.own].clone()
        for p in self.params:
            p.grad = None
        self.flat_grads = None

    def held_bytes(self):
        """Bytes of model states this rank holds right now."""
        size = lambda t: 0 if t is None else t.numel() * t.element_size()
        return {"weights": size(self.flat_weights),
                "gradients": size(self.flat_grads) + size(self.grad_shard),
                "optimizer state": size(self.exp_avg) + size(self.exp_avg_sq)}

### Ring collectives

In a ring, rank $r$ sends only to rank $(r+1) \bmod d$. A **reduce-scatter** takes $d-1$ rounds; in each, every rank sends one chunk to its neighbour, which adds it to its copy, until rank $r$ holds the sum of chunk $r$. An **all-gather** passes the finished chunks around in $d-1$ more rounds. A rank sends $1/d$ of the buffer per round, so an all-reduce of $N$ numbers sends

$$2\,\frac{d-1}{d}\,N \;<\; 2N$$

numbers per rank, for any $d$.

In [ ]:
def ring_reduce_scatter(buffers, bytes_sent):
    """In place on one flat buffer per rank: afterwards, chunk r of buffers[r] is the sum over ranks of
    chunk r. On a real cluster each rank is its own process and this is one call to
    torch.distributed.reduce_scatter_tensor, which NCCL runs over the network."""
    d = len(buffers)
    assert len(buffers[0]) % d == 0, "pad each buffer to a multiple of the number of ranks"
    c = len(buffers[0]) // d
    for step in range(d - 1):
        # Rank r passes on chunk (r - step - 1) mod d: the partial sum it received in the last round.
        senders = [(r, (r - step - 1) % d) for r in range(d)]
        # Copy every message before applying any: on a real cluster all ranks send at the same time.
        messages = [buffers[r][k * c:(k + 1) * c].clone() for r, k in senders]
        for (r, k), message in zip(senders, messages):
            buffers[(r + 1) % d][k * c:(k + 1) * c].add_(message)      # the neighbour adds it to its copy
            bytes_sent[r] += message.numel() * message.element_size()


def ring_all_gather(buffers, bytes_sent):
    """In place: rank r starts with a finished chunk r and ends with every rank's finished chunk
    (torch.distributed.all_gather_into_tensor on a real cluster)."""
    d = len(buffers)
    assert len(buffers[0]) % d == 0, "pad each buffer to a multiple of the number of ranks"
    c = len(buffers[0]) // d
    for step in range(d - 1):
        senders = [(r, (r - step) % d) for r in range(d)]               # rank r passes on the chunk it received last
        messages = [buffers[r][k * c:(k + 1) * c].clone() for r, k in senders]
        for (r, k), message in zip(senders, messages):
            buffers[(r + 1) % d][k * c:(k + 1) * c].copy_(message)     # the neighbour overwrites its copy
            bytes_sent[r] += message.numel() * message.element_size()

A check on random numbers, then the all-reduce traffic per rank for growing $d$, in multiples of the buffer size $N$.

In [ ]:
d = 4
local = [torch.randn(10 * d) for _ in range(d)]                          # one buffer of N = 40 numbers per rank
total = torch.stack(local).sum(dim=0)                                    # what an all-reduce must produce
buffers = [t.clone() for t in local]
bytes_sent = [0] * d
ring_reduce_scatter(buffers, bytes_sent)
print("after the reduce-scatter, rank r holds the sum of chunk r:",
      all(torch.allclose(buffers[r][10 * r:10 * (r + 1)], total[10 * r:10 * (r + 1)]) for r in range(d)))
ring_all_gather(buffers, bytes_sent)
print("after the all-gather, every rank holds the whole sum:     ", all(torch.allclose(b, total) for b in buffers))

# Patarasuk and Yuan (2009) proved that, when nothing is compressed and only the ranks add numbers,
# no all-reduce can make every rank send less than 2(d-1)/d of the buffer.
rows = []
for d in [2, 4, 8, 16, 32]:
    n = 1_000 * d                                                        # a buffer that splits evenly
    buffers = [torch.randn(n) for _ in range(d)]
    bytes_sent = [0] * d
    ring_reduce_scatter(buffers, bytes_sent)
    ring_all_gather(buffers, bytes_sent)
    rows.append({"ranks d": d, "numbers sent per rank / N": bytes_sent[0] / 4 / n, "2(d-1)/d": 2 * (d - 1) / d})
print()
print(pd.DataFrame(rows).to_string(index=False))

### Three ways to take a step

The optimizer is AdamW, written by hand, with weights $\theta$, gradient $g_t$ at step $t$, learning rate $\eta$, decay rates $\beta_1, \beta_2$, a small $\epsilon$ and weight decay $\lambda$:

$$\begin{aligned} m_t &= \beta_1 m_{t-1} + (1-\beta_1)\,g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2, \\ \theta_t &= \theta_{t-1} - \eta\left(\frac{m_t/(1-\beta_1^t)}{\sqrt{v_t/(1-\beta_2^t)} + \epsilon} + \lambda\,\theta_{t-1}\right) \end{aligned}$$

Every operation is elementwise and rounds each number the same way wherever it sits, so updating a chunk gives exactly the numbers that updating the whole buffer gives for it.

In [ ]:
@torch.no_grad()
def adamw_update(weights, grad, exp_avg, exp_avg_sq, step, lr):
    """AdamW on a flat slice of the weights, in place. Every operation is elementwise and rounds each
    number once. add_(..., alpha=...) and lerp_ (which torch.optim.AdamW uses) are avoided: on a CPU
    their vectorised loop fuses a multiply and an add but the leftover numbers at the end of a slice
    do not, so a chunk and the whole buffer could round differently."""
    exp_avg.mul_(BETA1).add_((1 - BETA1) * grad)                         # m
    exp_avg_sq.mul_(BETA2).add_((1 - BETA2) * grad * grad)               # v
    m_hat = exp_avg / (1 - BETA1 ** step)                                # bias corrections
    v_hat = exp_avg_sq / (1 - BETA2 ** step)
    weights.mul_(1 - lr * WEIGHT_DECAY)                                  # decoupled weight decay
    weights.sub_(lr * m_hat / (v_hat.sqrt() + EPS))

In a step, every rank runs forward and backward on its micro-batch, and a reduce-scatter sums the gradients. Then:

- **DDP** all-gathers the summed gradient, and each rank updates the whole model: $d$ identical updates
- **ZeRO-1**: rank $r$ updates chunk $r$ with that chunk's $m$ and $v$, then the ranks all-gather the weights
- **ZeRO-2**: the same, but each rank first frees the gradient outside its chunk

`held_bytes` is recorded just before the update. Our ZeRO-2 briefly holds the whole gradient, as ZeRO-1 does; real ZeRO-2 reduce-scatters gradients in groups as the backward pass produces them, so it never does.

In [ ]:
class Cluster:
    """world_size simulated GPUs training one model with one strategy: "ddp", "zero1" or "zero2"."""

    def __init__(self, initial_state, world_size, strategy):
        self.d, self.strategy = world_size, strategy
        self.ranks = []
        for r in range(world_size):
            replica = CharGPT().to(DEVICE)
            replica.load_state_dict(initial_state)                       # every replica starts from the same weights
            self.ranks.append(Rank(r, replica, world_size, strategy))
        self.bytes_sent = [0] * world_size                               # per rank, since the cluster was built
        self.held = None                                                 # per rank, at the last update


def train_step(cluster, inputs, targets, step, lr):
    """One data-parallel step on every rank of a cluster. Returns the loss averaged over the global batch."""
    losses = [rank.forward_backward(x, y) for rank, x, y in zip(cluster.ranks, inputs, targets)]
    # rank r: chunk r of the summed gradient (the list is built inline, so that no reference outlives
    # the call and ZeRO-2 can really free the full buffers)
    ring_reduce_scatter([rank.flat_grads for rank in cluster.ranks], cluster.bytes_sent)
    # Each rank's loss is a mean over its micro-batch, so the sum divided by d is the gradient of the
    # mean loss over the global batch.
    if cluster.strategy == "ddp":
        ring_all_gather([rank.flat_grads for rank in cluster.ranks], cluster.bytes_sent)   # every rank: the whole sum
        cluster.held = [rank.held_bytes() for rank in cluster.ranks]
        for rank in cluster.ranks:
            adamw_update(rank.flat_weights, rank.flat_grads / cluster.d,
                         rank.exp_avg, rank.exp_avg_sq, step, lr)
    else:
        if cluster.strategy == "zero2":
            for rank in cluster.ranks:
                rank.keep_own_gradient_only()
        cluster.held = [rank.held_bytes() for rank in cluster.ranks]
        for rank in cluster.ranks:                                       # each rank updates the chunk it owns
            own_grad = rank.grad_shard if cluster.strategy == "zero2" else rank.flat_grads[rank.own]
            adamw_update(rank.flat_weights[rank.own], own_grad / cluster.d,
                         rank.exp_avg, rank.exp_avg_sq, step, lr)
        ring_all_gather([rank.flat_weights for rank in cluster.ranks], cluster.bytes_sent)   # every rank: all updated chunks
    return torch.stack(losses).mean().item()

## 4. Training Three Clusters Side by Side

One cluster of `WORLD_SIZE` ranks per strategy, all starting from the model above and getting the same batches. The learning rate rises linearly for `WARMUP_STEPS` steps, then decays along a cosine. Every `EVAL_EVERY` steps we print the validation loss (of ZeRO-2's rank 0) and whether every replica has the same bits as DDP's rank 0.

Expected time: about 7 minutes on a T4.

In [ ]:
def lr_at(step):
    """Linear warm-up to LR over WARMUP_STEPS, then cosine decay to LR / 10 at MAX_STEPS."""
    if step <= WARMUP_STEPS:
        return LR * step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return LR * (0.1 + 0.45 * (1 + math.cos(math.pi * progress)))


STRATEGIES = {"ddp": "DDP", "zero1": "ZeRO-1", "zero2": "ZeRO-2"}
initial_state = {name: tensor.clone() for name, tensor in model.state_dict().items()}
clusters = {s: Cluster(initial_state, WORLD_SIZE, s) for s in STRATEGIES}
global_batch = WORLD_SIZE * MICRO_BATCH_SIZE
print(f"global batch: {WORLD_SIZE} ranks x {MICRO_BATCH_SIZE} windows x {CONTEXT} characters | "
      f"{MAX_STEPS:,} steps per cluster | one pass over the training text: "
      f"{len(train_data) // (global_batch * CONTEXT):,} steps")

batch_generator = torch.Generator().manual_seed(SEED)
history = {"step": [], "val_loss": [], "identical": [], "max_diff": [], **{s: [] for s in STRATEGIES}}
start = time.time()
for step in range(1, MAX_STEPS + 1):
    inputs, targets = get_batch(train_data, global_batch, batch_generator)
    inputs, targets = inputs.chunk(WORLD_SIZE), targets.chunk(WORLD_SIZE)   # rank r gets micro-batch r
    lr = lr_at(step)
    for s, cluster in clusters.items():
        history[s].append(train_step(cluster, inputs, targets, step, lr))
    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        reference = clusters["ddp"].ranks[0].flat_weights
        replicas = [rank.flat_weights for cluster in clusters.values() for rank in cluster.ranks]
        # Compare the raw 32-bit patterns: the same bits, not merely equal values (0.0 equals -0.0).
        identical = all(torch.equal(w.view(torch.int32), reference.view(torch.int32)) for w in replicas)
        max_diff = torch.stack([(w - reference).abs().max() for w in replicas]).max().item()
        val_loss = evaluate(clusters["zero2"].ranks[0].model)
        for key, value in zip(["step", "val_loss", "identical", "max_diff"], [step, val_loss, identical, max_diff]):
            history[key].append(value)
        losses = ", ".join(f"{name} {history[s][-1]:.4f}" for s, name in STRATEGIES.items())
        print(f"step {step}/{MAX_STEPS}: train loss {losses} | val loss {val_loss:.4f} | "
              f"{len(replicas)} replicas bit-identical: {identical} | {time.time() - start:.0f}s elapsed")

## 5. Results

### Did sharding change the training?

Training loss of the three clusters (dashed and dotted lines over the solid one); validation loss with the two baselines.

In [ ]:
steps = range(1, len(history["ddp"]) + 1)
fig, (ax_train, ax_val) = plt.subplots(1, 2, figsize=(13, 4.5))
for (s, name), style in zip(STRATEGIES.items(), ["-", "--", ":"]):
    ax_train.plot(steps, history[s], linestyle=style, linewidth=1.5, label=name)
ax_train.set(xlabel="step", ylabel="cross-entropy (nats per character)", title="Training loss")
ax_train.legend()
ax_val.plot(history["step"], history["val_loss"], marker="o", label="trained model")
ax_val.axhline(untrained_loss, color="grey", linestyle=":", label="untrained model")
ax_val.axhline(unigram_loss, color="grey", linestyle="--", label="character frequencies")
ax_val.set(xlabel="step", ylabel="cross-entropy (nats per character)", title="Validation loss")
ax_val.legend(loc="upper right", bbox_to_anchor=(1, 0.92))                # between the two baseline lines
plt.tight_layout()
plt.show()

print("training losses identical at every step:", all(history[s] == history["ddp"] for s in STRATEGIES))
print("replicas bit-identical at every evaluation:", all(history["identical"]))
print("largest weight difference at each evaluation:", history["max_diff"])

**What to notice**

The three curves are one curve and the weights stay bit-identical: ZeRO changed where the optimizer state and gradients live, not one number computed from them. (A real cluster's all-reduce and reduce-scatter may add numbers in different orders, so expect agreement up to rounding.) The validation loss falls far below both baselines.

### What each rank holds and sends

Rank 0 of each training cluster at its last update, ZeRO-3 by its formula (not simulated), and the traffic per step; then fresh clusters of 1, 2, 4 and 8 ranks after one step.

In [ ]:
W, G, K = 4, 4, 8                                                        # bytes per parameter in this lab
parts = ["weights", "gradients", "optimizer state"]
held = {name: clusters[s].held[0] for s, name in STRATEGIES.items()}
held["ZeRO-3 (formula)"] = dict(zip(parts, np.array([W, G, K]) * n_params / WORLD_SIZE))
held = pd.DataFrame(held).T[parts] / 1e6                                  # MB per rank
print(f"MB held by each of {WORLD_SIZE} ranks ({n_params:,} parameters):")
print(held.assign(total=held.sum(axis=1)).round(2).to_string())
for s, name in STRATEGIES.items():
    copies = clusters[s].bytes_sent[0] / (W * n_params * len(history[s]))   # per step actually taken
    print(f"{name}: rank 0 sent {copies:.3f} copies of the weights per step (2(d-1)/d = {2 * (WORLD_SIZE - 1) / WORLD_SIZE:.3f})")

rows = []
for d in [1, 2, 4, 8]:
    for s, name in STRATEGIES.items():
        cluster = Cluster(initial_state, d, s)
        inputs, targets = get_batch(train_data, d * MICRO_BATCH_SIZE, torch.Generator().manual_seed(SEED))
        train_step(cluster, inputs.chunk(d), targets.chunk(d), step=1, lr=LR)
        rows.append({"ranks": d, "strategy": name, "held per parameter": sum(cluster.held[0].values()) / n_params,
                     "sent per step / 4 Psi": cluster.bytes_sent[0] / (W * n_params)})
        del cluster
sweep = pd.DataFrame(rows)
print()
print(sweep.pivot(index="ranks", columns="strategy").round(3).to_string())

The measured points against the formulas (lines), with $W = G = 4$ and $K = 8$; traffic is in copies of the weights.

In [ ]:
ranks = np.array([1, 2, 4, 8])
formulas = {"DDP": np.full(len(ranks), W + G + K), "ZeRO-1": W + G + K / ranks,
            "ZeRO-2": W + (G + K) / ranks, "ZeRO-3 (formula)": (W + G + K) / ranks}
colors = {"DDP": "C0", "ZeRO-1": "C1", "ZeRO-2": "C2", "ZeRO-3 (formula)": "C3"}
markers = {"DDP": ("o", 90), "ZeRO-1": ("s", 50), "ZeRO-2": ("^", 25)}   # shrinking, so overlapping points stay visible
fig, (ax_bar, ax_mem, ax_net) = plt.subplots(1, 3, figsize=(18, 4.8))
held.plot.bar(stacked=True, ax=ax_bar, rot=0)
ax_bar.set(ylabel="MB per rank", title=f"Model states per rank, {WORLD_SIZE} ranks")
for name, line in formulas.items():
    ax_mem.plot(ranks, line, color=colors[name], linestyle=":" if "formula" in name else "-", label=name)
ax_net.plot(ranks, 2 * (ranks - 1) / ranks, color="grey", label="2(d-1)/d")
ax_net.axhline(2, color="grey", linestyle="--", linewidth=0.8, label="two copies, the limit as d grows")
ax_net.plot(ranks, 3 * (ranks - 1) / ranks, color=colors["ZeRO-3 (formula)"], linestyle=":",
            label="ZeRO-3 (formula), 3(d-1)/d")
for name, group in sweep.groupby("strategy"):
    marker, size = markers[name]
    ax_mem.scatter(group["ranks"], group["held per parameter"], color=colors[name], marker=marker, s=size, zorder=3)
    ax_net.scatter(group["ranks"], group["sent per step / 4 Psi"], color=colors[name], marker=marker, s=size,
                   zorder=3, label=f"{name}, measured")
for ax in (ax_mem, ax_net):
    ax.set_xscale("log", base=2)
    ax.set_xticks(ranks, [str(r) for r in ranks])
    ax.set_xlabel("ranks d")
    ax.legend()
ax_mem.set(ylabel="bytes per parameter on each rank", title="Model states per rank", ylim=(0, None))
ax_net.set(ylabel="sent per rank per step (copies of the weights)", title="Traffic per rank")
plt.tight_layout()
plt.show()

**What to notice**

The points sit on the formulas. ZeRO-1 and ZeRO-2 approach what they still replicate, $W + G$ and $W$ bytes per parameter; only ZeRO-3 keeps falling as $1/d$. All three send the same, approaching two copies of the weights per step; ZeRO-3 would send 1.5 times as much.

### Save, reload and generate

Under ZeRO-1 and ZeRO-2 every rank holds all the weights, so rank 0 can save the model; resuming training would also need every rank's shard of $m$ and $v$. The reloaded model then continues a prompt, next to the untrained model.

In [ ]:
# Clone each tensor, so that it is saved on its own rather than as a view of the flat buffer.
trained = clusters["zero2"].ranks[0].model
torch.save({name: t.clone() for name, t in trained.state_dict().items()}, "char_gpt_zero2.pth")
reloaded = CharGPT().to(DEVICE)
reloaded.load_state_dict(torch.load("char_gpt_zero2.pth", map_location=DEVICE))
print(f"validation loss: trained {evaluate(trained):.4f} | reloaded {evaluate(reloaded):.4f}")


@torch.no_grad()
def generate(model, prompt, max_new_tokens, temperature=0.8):
    """Continues the prompt by sampling one character at a time. Dividing the logits by a temperature
    below 1 sharpens the distribution, so the samples stay closer to the likeliest characters."""
    model.eval()
    generator = torch.Generator(device=DEVICE).manual_seed(SEED)
    ids = encode([prompt or "\n"])[None].to(DEVICE)                      # (1, len(prompt)); an empty prompt starts a line
    for _ in range(max_new_tokens):
        logits = model(ids[:, -CONTEXT:])[:, -1] / temperature           # scores for the next character
        next_id = torch.multinomial(logits.softmax(dim=-1), 1, generator=generator)
        ids = torch.cat([ids, next_id], dim=1)
    model.train()
    return decode(ids[0].tolist())


prompt = "Once upon a time, there was a little"
print("\nUntrained model:\n" + generate(model, prompt, MAX_NEW_TOKENS // 4))
print("\nReloaded model:\n" + generate(reloaded, prompt, MAX_NEW_TOKENS))

## 6. The Same Arithmetic for Real Models

`per_gpu_gb` applies the formulas with mixed-precision sizes, $W = G = 2$ and $K = 12$ ($M_{\text{DDP}} = 16\Psi$, $M_1 = 4\Psi + 12\Psi/d$, $M_2 = 2\Psi + 14\Psi/d$, $M_3 = 16\Psi/d$), to a 7B model on 8 GPUs and a 70B model on 8 to 64, in GB ($10^9$ bytes).

In [ ]:
def per_gpu_gb(n_params, gpus, stage, W=2, G=2, K=12):
    """Model states per GPU in GB for ZeRO stage 0 (DDP), 1, 2 or 3."""
    replicated, sharded = {0: (W + G + K, 0), 1: (W + G, K), 2: (W, G + K), 3: (0, W + G + K)}[stage]
    return (replicated + sharded / gpus) * n_params / 1e9


plans = pd.DataFrame([{"model": f"{p / 1e9:.0f}B", "GPUs": gpus,
                       **{name: per_gpu_gb(p, gpus, stage) for stage, name in enumerate(["DDP", "ZeRO-1", "ZeRO-2", "ZeRO-3"])}}
                      for p, gpus in [(7e9, 8), (70e9, 8), (70e9, 16), (70e9, 32), (70e9, 64)]])
print("Model states per GPU (GB), mixed precision with Adam:")
print(plans.round(2).to_string(index=False))

**What to notice**

Under DDP the 7B model needs over 80 GB per GPU, but any ZeRO stage fits it on 8 GPUs. The 70B model fits only under ZeRO-3, on enough GPUs to leave room for activations: earlier stages keep all the 16-bit weights, $2\Psi$ = 140 GB, on every GPU.

## Summary

- Data parallelism with Adam stores $16\Psi$ bytes of model states on every rank, whatever $d$ is.
- An all-reduce is a reduce-scatter plus an all-gather: under two copies of the buffer per rank on a ring.
- Since Adam is elementwise, ZeRO-1 and ZeRO-2 update each chunk between the two: less memory, the same traffic and, here, DDP's weights to the last bit.
- Only ZeRO-3, at 1.5 times the traffic, also shards the weights.

## Exercises

1. **Eight ranks.** Set `WORLD_SIZE = 8` and `MICRO_BATCH_SIZE = 8` (the same global batch) and `MAX_STEPS = 300`, and rerun from the config cell. Do the clusters still agree bit for bit? Which bars shrink, and does the printed traffic match $2(d-1)/d$?
2. **No all-gather.** Comment out the all-gather at the end of the ZeRO branch of `train_step`, set `MAX_STEPS = 50` and `EVAL_EVERY = 10`, and rerun. Which numbers of a ZeRO replica never change, and why does its training loss lag DDP's?
3. **Gradient clipping.** Add `MAX_GRAD_NORM = 1.0` to the config and scale the averaged gradient in `train_step` by $\min(1, c/\|\nabla\|_2)$, with $c$ = `MAX_GRAD_NORM` and the norm over all parameters. A ZeRO rank holds one chunk: which collective gives it the norm, and at what cost? Why might the clusters stop being bit-identical?
4. **Plan a run.** With `per_gpu_gb`, find for each stage the fewest 80 GB GPUs leaving 40 GB each for activations for a 13B model, or show that none suffice.

## Further Reading

- [Rajbhandari et al., 2020. ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054v3)
- [Patarasuk and Yuan, 2009. Bandwidth Optimal All-reduce Algorithms for Clusters of Workstations](https://doi.org/10.1016/j.jpdc.2008.09.002)
- [Zhao et al., 2023. PyTorch FSDP: Experiences on Scaling Fully Sharded Data Parallel](https://arxiv.org/abs/2304.11277v2)
- In PyTorch, [`ZeroRedundancyOptimizer`](https://docs.pytorch.org/docs/stable/distributed.optim.html) shards the optimizer state and [`fully_shard`](https://docs.pytorch.org/docs/stable/distributed.fsdp.fully_shard.html) (Fully Sharded Data Parallel) all three

### Contributed by: Ali Habibullah